# 🧪 How Differentiable NMR Structure Refinement Works

This notebook explains the key concepts behind `diff-integrator`: a differentiable
integrative NMR structure refinement engine built on JAX and Optax.

By the end you will understand:
- What NMR observables (chemical shifts, RDCs) actually measure and why they are useful
- How backbone dihedral angles are turned into 3D coordinates — and why two parameterisations exist
- Why automatic differentiation is uniquely powerful for combining multiple experimental constraints
- Why the Saupe alignment tensor must be fixed during gradient descent
- How geometry anchors, bond-length penalties, and Ramachandran priors prevent unphysical backbone geometries
- How per-term early stopping prevents wasted compute and late-stage structural drift

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/elkins-lab/diff-integrator.git
    %pip install -q ./diff-integrator matplotlib seaborn numpy
    import os
    os.chdir('diff-integrator/examples/interactive_tutorials')

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
RES = Path("../../benchmarks/results")

## 1. What Do NMR Observables Measure?

### Cα Chemical Shifts
The **Cα chemical shift** (δ_CA, measured in ppm) is the resonance frequency of the alpha-carbon nucleus in an NMR spectrum. It is highly sensitive to **local backbone geometry** because nearby electrons change the effective magnetic field at the carbon nucleus.

Key empirical fact: Cα chemical shifts differ systematically between secondary structures:
- **α-helix**: δ_CA is ~3 ppm *above* the random-coil value
- **β-sheet**: δ_CA is ~1.5 ppm *below* the random-coil value
- **Random coil**: baseline, defined per amino acid type

This makes Cα shifts powerful validators of the secondary structure propensity of a backbone conformation.

### Residual Dipolar Couplings (RDCs)
When a protein is weakly aligned in solution (using a medium like polyacrylamide gel or Pf1 phage), internuclear bonds are not perfectly averaged by rotation. The remaining dipolar coupling depends on the **orientation of the bond vector relative to the magnetic field**:
$$D_{NH} = D_{max} \left[ S_{zz}(3\cos^2\theta - 1) + (S_{xx} - S_{yy})\sin^2\theta\cos 2\phi \right] / 2$$
where $S_{ij}$ is the Saupe alignment tensor and $(\theta, \phi)$ are the spherical coordinates of the N–H bond.

RDCs constrain the **global orientation** of bond vectors — complementary to the local information from chemical shifts.

In [ ]:
aas = ['ALA', 'GLY', 'VAL', 'LEU', 'SER']
rc_vals = [52.5, 45.4, 62.2, 55.1, 58.3] # illustrative baselines

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(aas))
width = 0.25

ax.bar(x - width, [r - 1.5 for r in rc_vals], width, label='β-sheet (-1.5 ppm)', color='#3498db')
ax.bar(x, rc_vals, width, label='Random Coil (baseline)', color='#95a5a6')
ax.bar(x + width, [r + 3.0 for r in rc_vals], width, label='α-helix (+3.0 ppm)', color='#e74c3c')

ax.set_ylabel('Cα Chemical Shift (ppm)')
ax.set_title('Illustrative Cα Shift Offsets by Secondary Structure')
ax.set_xticks(x)
ax.set_xticklabels(aas)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Optimizing Over Backbone Coordinates — Two Parameterisations

### 2a. Dihedral-Angle (NeRF) Parameterisation

Each residue has two backbone dihedral angles:
- **φ (phi)**: rotation around the N–Cα bond
- **ψ (psi)**: rotation around the Cα–C bond

A NeRF (Natural Extension Reference Frame) builder reconstructs full 3D Cartesian
coordinates from (φ, ψ) arrays using ideal bond lengths (1.46 Å N–Cα, 1.52 Å Cα–C)
and angles (111° Cα–N–C).

**Advantage:** the optimizer only needs to move in a low-dimensional, physically meaningful
space. Bond lengths and angles are automatically satisfied by construction.

**Limitation:** ideal-geometry reconstruction accumulates rounding errors over long chains.
For proteins > 80–90 residues, the NeRF-rebuilt starting structure can deviate ~14 Å from
the raw PDB Cα trace (HR2876B, 107 residues), creating a distorted gradient landscape.

### 2b. Cartesian Parameterisation (new in 2025)

For longer proteins, `diff-integrator` can now optimise **directly in Cartesian Cα
coordinate space**, starting from the raw PDB atomic positions.  There is no NeRF builder
in this path, so there is zero geometric drift.  Instead, bond-length and bond-angle
deviations from ideal Engh & Huber values are penalised as soft harmonic losses:

$$L_{bond} = \frac{1}{N-1} \sum_{i} (b_i - b_0)^2$$
$$L_{angle} = \frac{1}{N-2} \sum_{i} (\theta_i - \theta_0)^2$$

`CartesianCAShiftLoss` extracts φ/ψ angles on-the-fly from the current coordinates using
the differentiable `compute_phi_psi` function, so chemical shifts can still be predicted
and back-differentiated:

```python
from diff_integrator.terms.chemical_shifts import CartesianCAShiftLoss
from diff_integrator.terms.bond_geometry import make_backbone_bond_geometry

# Bond-length + bond-angle harmonic penalties (stiff, weight 10–50)
bond_len, bond_ang = make_backbone_bond_geometry(res_names)

# Cartesian shift loss — extracts phi/psi from coords on-the-fly
ca_shift = CartesianCAShiftLoss(exp_res_ids, exp_shifts, struct_res_ids, res_names)

joint_loss = JointLoss([
    (position_anchor, 5.0),   # decays via ExponentialDecaySchedule
    (ca_shift, 1.0),
    (bond_len, 50.0),
    (bond_ang, 10.0),
])
```

**HR2876B result (107 residues):** Cartesian approach achieved **13× larger** Cα RMSD
improvement and **30× less** structural drift than the NeRF approach.

In [ ]:
try:
    phi_psi = np.load(RES / '2KZV/phi_psi_init.npy')
    phi_deg, psi_deg = phi_psi[:, 0], phi_psi[:, 1]

    fig, ax = plt.subplots(figsize=(7, 7))
    
    # Helix region
    rect_helix = mpatches.Rectangle((-170, -100), 140, 145, fill=True, color='lightgray', alpha=0.5)
    ax.add_patch(rect_helix)
    ax.text(-100, -25, 'α-helix', ha='center', color='gray')

    # Sheet region
    rect_sheet1 = mpatches.Rectangle((-170, 90), 115, 90, fill=True, color='lightgray', alpha=0.5)
    rect_sheet2 = mpatches.Rectangle((-170, -180), 115, 15, fill=True, color='lightgray', alpha=0.5)
    ax.add_patch(rect_sheet1)
    ax.add_patch(rect_sheet2)
    ax.text(-110, 135, 'β-sheet', ha='center', color='gray')

    ax.scatter(phi_deg, psi_deg, color='#2ecc71', edgecolors='k', alpha=0.8, s=40)

    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xticks(np.arange(-180, 181, 60))
    ax.set_yticks(np.arange(-180, 181, 60))
    ax.set_xlabel('Φ (degrees)')
    ax.set_ylabel('Ψ (degrees)')
    ax.set_title('Ramachandran Plot — 2KZV Initial Structure')
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.show()
except FileNotFoundError:
    print("2KZV phi_psi data not found.")

## 3. The Gradient Descent Loop

The core optimization in `diff-integrator` follows this pattern.  Two modes are shown:

### Mode A — Dihedral (NeRF) refinement

```python
for epoch in range(epochs):
    # 1. Build Cartesian coordinates from current angles
    coords = nerf_builder(phi, psi)           # (3N, 3) backbone atoms

    # 2. Evaluate all loss terms
    loss = (
        weight_geom  * geometry_loss(coords) +    # anchor to initial
        weight_shift * ca_shift_loss(phi, psi) +  # BMRB chemical shifts
        weight_rdc   * rdc_loss(coords)            # fixed-tensor RDC MSE
    )

    # 3. Auto-differentiate through the entire forward model
    grad_phi, grad_psi = jax.grad(loss)(phi, psi)

    # 4. Adam optimizer update
    phi, psi = optax.apply_updates((phi, psi), updates)

    # 5. (Every 50 epochs) Re-fit Saupe tensor outside gradient
    saupe_tensor = fit_saupe_tensor(coords)
```

### Mode B — Cartesian refinement (for long chains)

```python
from diff_integrator.optimizer import EarlyStopping, IntegrativeRefiner

result = IntegrativeRefiner(joint_loss).run(
    init_params=raw_pdb_coords,    # (3N, 3) — no NeRF builder needed
    epochs=2000,
    kinematics_fn=None,            # identity: params ARE the coords
    weight_schedules={0: anchor_schedule},   # anchor decays 5.0 → 0.1

    # Per-term early stopping: exit when Cα shift RMSD stops improving
    early_stopping=EarlyStopping(
        term_index=1,     # index of CartesianCAShiftLoss in JointLoss
        patience=75,      # stop if no improvement for 75 consecutive epochs
        min_delta=5e-5,   # ~0.05 mppm threshold (unweighted, in natural units)
    ),
)
print(f"Stopped at epoch {result.stopped_at_epoch} — {result.early_stopping_triggered_by}")
# HR2876B: "Stopped at epoch 894 — term_1 (ca_shift) patience=75"
```

Because JAX can differentiate through the NeRF builder, shift predictor, bond-length
calculator, and RDC back-calculator, gradients flow all the way back to the optimised
parameters from every experimental observable simultaneously.

In [ ]:
epochs = np.linspace(0, 500, 100)
loss_geom = 2.0 * np.exp(-epochs/100) + 0.5
loss_shift = 1.5 * np.exp(-epochs/50) + 0.2
loss_rdc = 3.0 * np.exp(-epochs/150) + 0.8
total_loss = loss_geom + loss_shift + loss_rdc

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, loss_geom, '--', label='Geometry Loss (anchor)', color='#9b59b6')
ax.plot(epochs, loss_shift, '--', label='Cα Shift Loss', color='#3498db')
ax.plot(epochs, loss_rdc, '--', label='RDC Loss', color='#e67e22')
ax.plot(epochs, total_loss, '-', label='Total Loss', color='black', linewidth=3)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss Value')
ax.set_title('Schematic: Multi-Objective Loss Descent')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Why the Saupe Tensor Must Be Fixed During Gradient Descent

This is the most subtle but critical design decision in `diff-integrator`.

### The Degeneracy Trap
The Saupe tensor $S$ has **5 free parameters** ($D_a$, $R$, and 3 Euler angles). Given any set of bond vectors, there always exists an alignment tensor that perfectly fits those RDCs — this is a linear least squares problem.

If we allow the optimizer to **differentiate through the tensor fitting** (i.e., $S$ is a function of the current coordinates), the optimizer discovers this exploit: instead of moving the backbone toward a physically correct conformation, it learns to rotate bond vectors in a degenerate way that satisfies the tensor equations without improving the real structure.

The result: **Q → 0, but the structure becomes physically impossible** — a tangled "pretzel" that satisfies the orientation constraints but has zero resemblance to a real protein.

### The Solution: `FixedTensorRDCLoss`
`diff-integrator` uses `jax.lax.stop_gradient` to freeze the tensor during backpropagation. The tensor is re-fitted every 50 epochs (outside the gradient computation) to track the evolving backbone. This is the same protocol used by X-PLOR, CNS, and PALES.

```python
# WRONG (exploitable degeneracy):
loss = rdc_mse(coords, fit_tensor(coords))  # gradient flows through fit_tensor

# CORRECT (FixedTensorRDCLoss):
fixed_tensor = jax.lax.stop_gradient(fit_tensor(coords))  # tensor is frozen
loss = rdc_mse(coords, fixed_tensor)                       # gradient only flows through coords
```

In [ ]:
epochs = np.linspace(0, 500, 100)
# Degenerate model plummets to 0 quickly
q_degenerate = 0.4 * np.exp(-epochs/40)
# Fixed tensor improves gradually to a physical plateau
q_fixed = 0.4 * np.exp(-epochs/150) + 0.15

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, q_degenerate, '-', color='#e74c3c', linewidth=2, label='Differentiating through tensor (Exploits degeneracy)')
ax.plot(epochs, q_fixed, '-', color='#2ecc71', linewidth=3, label='Fixed tensor protocol (Physically constrained)')

ax.axhspan(0.1, 0.35, color='gray', alpha=0.2, label='Physically reasonable Q range')

ax.annotate('Structure unravelling into\nnon-physical "pretzel"', xy=(150, 0.05), xytext=(200, 0.1),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6))

ax.set_xlabel('Epoch')
ax.set_ylabel('RDC Q-Factor')
ax.set_title('Synthetic Demo: With vs. Without Fixed Tensor')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Preventing Unphysical Backbone Geometries

Experimental restraints alone define only a subset of the backbone degrees of freedom.
Three complementary regularisers are used to keep the structure physically realistic:

### 5a. Position Anchor (`GeometryLoss`)

A harmonic spring restrains each backbone atom to its initial position:
$$L_{geom} = \frac{1}{3N} \sum_i \|\mathbf{r}_i - \mathbf{r}_i^{(0)}\|^2$$

For long-chain Cartesian refinement, the weight is **annealed** from strong (5.0) to weak
(0.1) over ~200 epochs via `ExponentialDecaySchedule`, allowing the experimental gradient
to gain influence as the bond/angle penalties tighten.

### 5b. Bond-Length and Bond-Angle Penalties

In Cartesian mode, explicit harmonic penalties enforce Engh & Huber ideal geometry
throughout optimisation:

```python
bond_len_loss, bond_ang_loss = make_backbone_bond_geometry(residue_names)
# HR2876B at convergence: bond RMSD = 0.0007 Å, angle RMSD = 0.33°
```

These are **stiff** (weight 10–50) so the backbone stays geometrically valid even when
the position anchor has relaxed.

### 5c. Sequence-Aware Ramachandran Prior (`RamachandranLoss`)

A soft prior penalises non-physical (φ, ψ) regions using residue-specific Gaussian
potential wells trained on the PDB:

- **Standard residues**: twin wells for α-helix and β-sheet favoured regions
- **Glycine**: additional ε-basin well (φ > 0), which is unique to GLY's lack of Cβ
- **Proline**: ring constraint restricts φ to −75° ± 20°, disallowing positive φ values

```python
from diff_integrator.terms.ramachandran import RamachandranLoss

rama = RamachandranLoss(residue_types=res_names)  # sequence-aware weighting
# Weight = 0.5; acts as a soft prior throughout training
```

This was absent in the original implementation and caused the optimizer to over-penalise
GLY residues in the ε-basin.  Adding it to the GmR58A and 2KZV benchmarks improved
Q-factor results significantly (−82% PAG Q on 2KZV).

## 6. Per-Term Early Stopping

A classic pitfall in multi-objective refinement is that the **total loss** continues to
decrease (due to the geometry anchor or bond penalties) long after the experimental
observable of interest has plateaued.  The optimizer then wastes compute — or worse,
distorts geometry while the anchor is weak.

`EarlyStopping` solves this by monitoring a single `JointLoss` term's **unweighted**
value independently:

| Design choice | Rationale |
|---|---|
| Unweighted monitoring | Threshold is in natural units (ppm, Å²), independent of weight schedules |
| Per-term, not total loss | Allows the geometry anchor to keep changing while the observable is watched |
| `mode="min"` or `"max"` | Extensible to scores (e.g., a correlation coefficient) as well as losses |
| Best-checkpoint preserved | `result.best_params` always holds the best iterate regardless of when stopping fires |

The result fields make it easy to audit the run:

```python
result.stopped_early            # True
result.stopped_at_epoch         # 894
result.early_stopping_triggered_by  # "term_1 (ca_shift) patience=75"
result.best_epoch               # 894  (coincides with stopping epoch here)
```

On HR2876B Cartesian (epochs=2000, patience=75), early stopping fired at epoch 894,
**saving 55% of the planned compute** while achieving the same Cα RMSD as the full run.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Initial
x1, y1 = [0, 1, 2, 2, 1], [0, 1, 1, 2, 3]
ax1.plot(x1, y1, 'o-', color='black', markersize=8, linewidth=2)
ax1.set_title('1. Initial Structure')

# Unanchored (satisfies vectors, loses shape)
x2, y2 = [0, 1.2, 1.2, 0, -1.2], [0, 1.2, 2.5, 3.5, 4.5]
ax2.plot(x2, y2, 'o-', color='#e74c3c', markersize=8, linewidth=2)
ax2.set_title('2. No Geometry Anchor\n(Vectors align, fold destroyed)')

# Anchored
x3, y3 = [0, 0.9, 1.8, 1.9, 0.9], [0, 1.1, 1.1, 2.1, 2.9]
ax3.plot(x3, y3, 'o-', color='#2ecc71', markersize=8, linewidth=2)
ax3.plot(x1, y1, 'o--', color='gray', alpha=0.5, label='Initial anchor')
for i in range(len(x1)):
    ax3.arrow(x1[i], y1[i], x3[i]-x1[i], y3[i]-y1[i], head_width=0.05, color='gray', alpha=0.5)
ax3.set_title('3. With Geometry Anchor\n(Local tweaks, fold preserved)')
ax3.legend()

for ax in (ax1, ax2, ax3):
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1, 5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

plt.tight_layout()
plt.show()